# Chương 3 — Market Basket Analysis (Google Colab — Pure Julia)

Phiên bản dành riêng cho **Google Colab (Julia Runtime)** — tự động `git clone` repository, tải dữ liệu `Groceries` khi chạy lần đầu, gọi trực tiếp `negFIN` tự viết của nhóm và hiển thị biểu đồ SVG ngay trên Colab.

### Hướng dẫn chạy trên Google Colab
1. Vào menu **Runtime $\rightarrow$ Change runtime type $\rightarrow$ chọn `Julia`** $\rightarrow$ bấm **Save**.
2. (Tùy chọn) Chỉnh các biến trong khối **SETUP & CONFIG** ở Phần 1 (`ALGO`, `MARKET_MINSUP`, `MARKET_MINCONF`, `REPO_BRANCH`).
3. Bấm **Runtime $\rightarrow$ Run all** để chạy kiểm thử Chương 3, khai thác luật kết hợp và vẽ biểu đồ Top-10 `Lift`.
4. Để chia sẻ nộp bài: **File $\rightarrow$ Save a copy in Drive** $\rightarrow$ **Share $\rightarrow$ Anyone with the link (Viewer)**.

### Mục lục các phần trong Notebook
- **Phần 1:** Thiết lập môi trường (`SETUP & CONFIG`), tự động `git clone` và nạp module `Applications`.
- **Phần 2:** Chạy bộ kiểm thử tự động Chương 3 (`tests/test_ch3_julia.jl`).
- **Phần 3:** Chạy phân tích giỏ hàng `Groceries`, in Top-10 luật theo `Lift` và hiển thị biểu đồ SVG.

---
## Phần 1: Thiết Lập Môi Trường & Nạp Module `Applications`

Cell này tự động xác định thư mục dự án (hoặc `git clone` từ GitHub khi mở độc lập trên Colab), kích hoạt `Project.toml` và nạp `src/applications/applications.jl`.

In [ ]:
using Pkg: Pkg

# ==============================================================================
# SETUP & CONFIG (CHỈNH THAM SỐ TẠI ĐÂY NẾU CẦN)
# ==============================================================================
const GITHUB_REPO_URL = "https://github.com/Thanhtung46/Frequent_Itemset_Mining.git"
const REPO_BRANCH = "main"
const ALGO = :optimized       # :optimized hoặc :baseline
const MARKET_MINSUP = 0.01    # Ngưỡng support tối thiểu (1%)
const MARKET_MINCONF = 0.15   # Ngưỡng confidence tối thiểu (15%)

# Xác định thư mục gốc dự án (tự động git clone nếu mở độc lập trên Colab)
function resolve_project_root()::String
    dir = pwd()
    for _ in 1:4
        if isfile(joinpath(dir, "src", "applications", "applications.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && break
        dir = parent
    end
    clone_dir = joinpath(pwd(), "Frequent_Itemset_Mining")
    if !isdir(clone_dir)
        println("Đang clone mã nguồn dự án từ GitHub cho môi trường Colab...")
        run(`git clone --depth 1 --branch $REPO_BRANCH $GITHUB_REPO_URL $clone_dir`)
    end
    return clone_dir
end

const ROOT_DIR = resolve_project_root()
Pkg.activate(ROOT_DIR)
include(joinpath(ROOT_DIR, "src", "applications", "applications.jl"))
using .Applications: run_market_basket

println("Julia version: ", VERSION)
println("Project root:  ", ROOT_DIR)

---
## Phần 2: Chạy Kiểm Thử Tự Động Chương 3 (`tests/test_ch3_julia.jl`)

Kiểm tra tích hợp trực tiếp với `negFIN`, độ chính xác `support`/`confidence`/`lift` và tính tái lập của toàn bộ pipeline Chương 3.

In [ ]:
include(joinpath(ROOT_DIR, "tests", "test_ch3_julia.jl"))

---
## Phần 3: Chạy Phân Tích Giỏ Hàng `Groceries` & Hiển Thị Kết Quả

- Dữ liệu: **9,835 giao dịch**, **169 nhóm sản phẩm** (tự động tải vào `data/chapter3/groceries.csv` khi chạy lần đầu).
- Mọi tập phổ biến đều được đối chiếu lại trên dữ liệu gốc bằng `validate_support`.
- Lưu ý: `Lift > 1` chỉ phản ánh **tương quan thống kê dương**, không chứng minh quan hệ nhân quả.

In [ ]:
market_res = run_market_basket(;
    minsup=MARKET_MINSUP,
    minconf=MARKET_MINCONF,
    algo=ALGO,
)

market_dir = joinpath(ROOT_DIR, "results", "chapter3", "market_basket")
println("\n=== MARKET BASKET SUMMARY ===")
println(read(joinpath(market_dir, "summary.txt"), String))
println("=== TOP-10 LUẬT THEO LIFT ===")
for line in Iterators.take(eachline(joinpath(market_dir, "top10_lift.csv")), 11)
    println(line)
end

display("image/svg+xml", read(joinpath(market_dir, "top10_lift.svg"), String))